In [ ]:
from pathlib import Path
import json,sys
dest=Path('/kaggle/working/calibration_code');dest.mkdir(exist_ok=True)
sources={'predict.py': '"""Run the exported research classifier on an image. No training or downloads."""\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\n\n\ndef load_config(directory):\n    directory = Path(directory).resolve()\n    config = json.loads((directory / \'inference_config.json\').read_text())\n    if config.get(\'architecture\') not in {\'efficientnet_b0\', \'densenet121\', \'resnet18\'}:\n        raise ValueError(\'Unsupported architecture\')\n    if config.get(\'classes\') != [\'NORMAL\', \'PNEUMONIA\']:\n        raise ValueError(\'Unexpected class mapping\')\n    if config.get(\'image_size\', 224) not in {224, 320}:\n        raise ValueError(\'Unsupported image size\')\n    threshold = config.get(\'threshold\')\n    if not isinstance(threshold, (int, float)) or not 0 <= threshold <= 1:\n        raise ValueError(\'Invalid threshold\')\n    names = config.get(\'selected_checkpoints\', [])\n    temperature = config.get(\'temperature\', 1.0)\n    if not isinstance(temperature, (int, float)) or not math.isfinite(temperature) or temperature <= 0:\n        raise ValueError(\'Invalid temperature\')\n    if not names or len(names) != len(set(names)):\n        raise ValueError(\'Missing or repeated checkpoints\')\n    paths = []\n    for name in names:\n        path = (directory / name).resolve()\n        if not path.is_relative_to(directory) or not path.is_file():\n            raise ValueError(\'Checkpoint must exist inside the model directory\')\n        paths.append(path)\n    return config, paths\n\n\ndef predict(directory, image_path, device_name=\'cpu\'):\n    import torch\n    from PIL import Image, ImageOps\n    from torchvision import models, transforms\n    config, paths = load_config(directory)\n    device = torch.device(device_name)\n    image_size = config.get(\'image_size\', 224)\n    with Image.open(image_path) as image:\n        image = ImageOps.pad(image.convert(\'RGB\'), (image_size, image_size),\n                             method=Image.Resampling.BICUBIC, color=0)\n    transform = transforms.Compose([transforms.ToTensor(),\n        transforms.Normalize([.485, .456, .406], [.229, .224, .225])])\n    x = transform(image).unsqueeze(0).to(device)\n    probabilities = []\n    with torch.inference_mode():\n        for path in paths:\n            if config[\'architecture\'] == \'efficientnet_b0\':\n                model = models.efficientnet_b0(weights=None)\n                model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 1)\n            elif config[\'architecture\'] == \'densenet121\':\n                model = models.densenet121(weights=None)\n                model.classifier = torch.nn.Linear(model.classifier.in_features, 1)\n            else:\n                model = models.resnet18(weights=None)\n                model.fc = torch.nn.Linear(model.fc.in_features, 1)\n            model.load_state_dict(torch.load(path, map_location=\'cpu\', weights_only=True))\n            model.to(device).eval()\n            probabilities.append(float(torch.sigmoid(model(x).flatten())[0].cpu()))\n            del model\n    probability = sum(probabilities) / len(probabilities)\n    raw_probability = probability\n    if \'temperature\' in config:\n        p = min(1 - 1e-7, max(1e-7, probability))\n        z = (math.log(p) - math.log1p(-p)) / config[\'temperature\']\n        probability = 1 / (1 + math.exp(-z)) if z >= 0 else math.exp(z) / (1 + math.exp(z))\n    return {\'image\': str(image_path), \'pneumonia_score\': probability,\n            \'threshold\': config[\'threshold\'],\n            \'predicted_label\': config[\'classes\'][int(probability >= config[\'threshold\'])],\n            \'member_scores\': probabilities,\n            \'raw_mean_score\': raw_probability,\n            \'temperature\': config.get(\'temperature\', 1.0),\n            \'image_size\': image_size,\n            \'note\': \'Research image classification score, not a calibrated clinical probability or diagnosis.\'}\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--model-dir\', type=Path, required=True)\n    parser.add_argument(\'--image\', type=Path, required=True)\n    parser.add_argument(\'--device\', choices=[\'cpu\', \'cuda\'], default=\'cpu\')\n    args = parser.parse_args()\n    print(json.dumps(predict(args.model_dir, args.image, args.device), indent=2))\n', 'validate_artifacts.py': '"""Validate saved experiment consistency without loading PyTorch or images."""\nimport argparse\nimport csv\nimport json\nimport math\nfrom itertools import combinations\nfrom pathlib import Path\nfrom predict import load_config\n\n\ndef read_csv(path):\n    with path.open(newline=\'\', encoding=\'utf-8\') as file:\n        return list(csv.DictReader(file))\n\n\ndef validate(directory):\n    directory = Path(directory)\n    metrics = json.loads((directory / \'metrics.json\').read_text())\n    manifest = read_csv(directory / \'split_manifest.csv\')\n    split_names = {r[\'split\'] for r in manifest}\n    if split_names not in [{\'train\', \'validation\', \'test\'}, {\'train\', \'selection\', \'calibration\', \'test\'}]:\n        raise ValueError(\'Unexpected or missing manifest split\')\n    for required in ({\'test\', \'validation\'} if \'validation\' in split_names else {\'test\'}):\n        if required not in metrics:\n            raise ValueError(\'Missing reported metrics: \' + required)\n    for field in [\'path\', \'sha256\']:\n        if len({r[field] for r in manifest}) != len(manifest):\n            raise ValueError(f\'Duplicate manifest {field}\')\n    if any(not r[field] for r in manifest for field in [\'path\', \'patient\', \'sha256\']):\n        raise ValueError(\'Missing manifest identity field\')\n    partitions = {name: [r for r in manifest if r[\'split\'] == name]\n                  for name in sorted(split_names)}\n    for name, rows in partitions.items():\n        if not rows or {r[\'label\'] for r in rows} != {\'0\', \'1\'}:\n            raise ValueError(f\'{name} must have both classes\')\n    for field in [\'patient\', \'sha256\']:\n        sets = {name: {r[field] for r in rows} for name, rows in partitions.items()}\n        for a, b in combinations(partitions, 2):\n            if sets[a] & sets[b]:\n                raise ValueError(f\'{field} overlap between {a} and {b}\')\n    threshold = metrics[\'threshold\']\n    if not 0 <= threshold <= 1:\n        raise ValueError(\'Threshold outside probability range\')\n    confusion = {}\n    for name in sorted(split_names - {\'train\'}):\n        rows = read_csv(directory / f\'{name}_predictions.csv\')\n        expected = {(r[\'path\'], r[\'label\'], r[\'patient\'], r[\'sha256\']) for r in partitions[name]}\n        observed = {(r[\'path\'], r[\'y_true\'], r[\'patient\'], r[\'sha256\']) for r in rows}\n        if expected != observed or len(rows) != len(partitions[name]):\n            raise ValueError(f\'{name} predictions do not match the split manifest\')\n        tn = fp = fn = tp = 0\n        for row in rows:\n            score = float(row[\'probability\'])\n            y, pred = int(row[\'y_true\']), int(row[\'prediction\'])\n            if not math.isfinite(score) or not 0 <= score <= 1:\n                raise ValueError(\'Non-finite or invalid prediction score\')\n            if pred != int(score >= threshold):\n                raise ValueError(\'Prediction disagrees with saved threshold\')\n            tn += y == 0 and pred == 0\n            fp += y == 0 and pred == 1\n            fn += y == 1 and pred == 0\n            tp += y == 1 and pred == 1\n        calculated = {\'accuracy\': (tn+tp)/len(rows), \'sensitivity\': tp/(tp+fn),\n                      \'specificity\': tn/(tn+fp)}\n        if name in metrics:\n            for metric, value in calculated.items():\n                if not math.isclose(value, metrics[name][metric], abs_tol=1e-8):\n                    raise ValueError(f\'{name} {metric} does not match predictions\')\n        confusion[name] = {\'tn\': tn, \'fp\': fp, \'fn\': fn, \'tp\': tp}\n    if (directory / \'inference_config.json\').exists():\n        config, paths = load_config(directory)\n        if config[\'threshold\'] != threshold:\n            raise ValueError(\'Inference and evaluation thresholds disagree\')\n        if config.get(\'temperature\', 1) != metrics.get(\'temperature\', 1):\n            raise ValueError(\'Inference and evaluation temperatures disagree\')\n        checkpoints = [str(p.relative_to(directory.resolve())) for p in paths]\n    else:\n        checkpoints = [\'pneumonia_model.pt\', \'best_weights.pt\']\n        if not all((directory / name).is_file() for name in checkpoints):\n            raise ValueError(\'Baseline weights missing\')\n    return {\'validated\': True, \'counts\': {k: len(v) for k, v in partitions.items()},\n            \'confusion_matrices\': confusion, \'checkpoints\': checkpoints}\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'directory\', type=Path)\n    args = parser.parse_args()\n    print(json.dumps(validate(args.directory), indent=2))\n', 'validate_benchmark.py': '"""Reconstruct a completed benchmark\'s selection and frozen test predictions."""\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import expit, logit\nfrom sklearn.metrics import average_precision_score, log_loss, roc_auc_score\n\nfrom validate_artifacts import validate\n\n\ndef close(a, b, description, tolerance=1e-8):\n    if not math.isfinite(float(a)) or not math.isclose(float(a), float(b), rel_tol=1e-7, abs_tol=tolerance):\n        raise ValueError(\'Inconsistent \' + description)\n\n\ndef scores(path, expected, threshold):\n    frame = pd.read_csv(path).sort_values(\'sha256\').reset_index(drop=True)\n    if len(frame) != len(expected) or frame.sha256.duplicated().any():\n        raise ValueError(\'Duplicate or missing predictions: \' + str(path))\n    for field in [\'sha256\', \'patient\', \'path\']:\n        if not frame[field].equals(expected[field]):\n            raise ValueError(\'Prediction identity mismatch: \' + field)\n    if not np.array_equal(frame.y_true, expected.label):\n        raise ValueError(\'Prediction labels mismatch\')\n    p = frame.probability.to_numpy(float)\n    if not np.isfinite(p).all() or not ((p >= 0) & (p <= 1)).all():\n        raise ValueError(\'Invalid probability\')\n    if not np.array_equal(frame.prediction, (p >= threshold).astype(int)):\n        raise ValueError(\'Prediction disagrees with threshold\')\n    return p\n\n\ndef check_metrics(y, p, cutoff, reported):\n    predicted = p >= cutoff\n    tn = int(((y == 0) & ~predicted).sum()); fp = int(((y == 0) & predicted).sum())\n    fn = int(((y == 1) & ~predicted).sum()); tp = int(((y == 1) & predicted).sum())\n    actual = dict(auroc=roc_auc_score(y,p), average_precision=average_precision_score(y,p),\n                  accuracy=np.mean(y==predicted), sensitivity=tp/(tp+fn), specificity=tn/(tn+fp),\n                  brier_score=np.mean((y-p)**2), log_loss=log_loss(y,p,labels=[0,1]),\n                  tn=tn, fp=fp, fn=fn, tp=tp)\n    for name,value in actual.items():\n        close(reported[name],value,name)\n\n\ndef temperature_scale(p, temperature):\n    if not math.isfinite(temperature) or temperature <= 0:\n        raise ValueError(\'Invalid temperature\')\n    return expit(logit(np.clip(p,1e-7,1-1e-7))/temperature)\n\n\ndef validate_benchmark(directory):\n    root=Path(directory)\n    basic=validate(root)\n    manifest=pd.read_csv(root/\'split_manifest.csv\')\n    frames={k:f.sort_values(\'sha256\').reset_index(drop=True) for k,f in manifest.groupby(\'split\')}\n    config=json.loads((root/\'inference_config.json\').read_text())\n    results=json.loads((root/\'all_trial_results.json\').read_text())\n    expected={(a,s) for a in [\'efficientnet_b0\',\'densenet121\',\'resnet18\'] for s in [42,123,2026]}\n    if len(results)!=9 or {(r[\'architecture\'],r[\'seed\']) for r in results}!=expected:\n        raise ValueError(\'Incomplete or duplicate architecture/seed trials\')\n    selection_by_arch={a:[] for a,_ in expected}; raw_test={}; raw_selection={}\n    for record in results:\n        trial=f"{record[\'architecture\']}_seed_{record[\'seed\']}"\n        if record[\'trial\']!=trial:\n            raise ValueError(\'Unexpected trial identifier\')\n        folder=root/trial\n        if not (folder/\'best_weights.pt\').is_file():\n            raise ValueError(\'Missing trial weights: \'+trial)\n        history=pd.read_csv(folder/\'history.csv\')\n        if len(history)!=record[\'epochs_completed\'] or list(history.epoch)!=list(range(1,len(history)+1)):\n            raise ValueError(\'Inconsistent epoch history\')\n        best=-1.; chosen=None\n        for row in history.itertuples():\n            if row.selection_auroc>best+1e-5:\n                best=row.selection_auroc; chosen=row.epoch\n        if chosen!=record[\'best_epoch\']:\n            raise ValueError(\'Checkpoint selection does not follow recorded rule\')\n        ps=scores(folder/\'selection_predictions.csv\',frames[\'selection\'],.5)\n        auc=float(roc_auc_score(frames[\'selection\'].label,ps))\n        close(auc,record[\'selection_auroc\'],\'selection AUROC\')\n        close(auc,best,\'checkpoint AUROC\',tolerance=1e-5)\n        selection_by_arch[record[\'architecture\']].append(auc)\n        raw_selection[trial]=ps\n        p=scores(folder/\'test_predictions_raw.csv\',frames[\'test\'],record[\'raw_threshold\'])\n        q=scores(folder/\'test_predictions.csv\',frames[\'test\'],record[\'threshold\'])\n        if not np.allclose(q,temperature_scale(p,record[\'temperature\']),atol=1e-10,rtol=1e-8):\n            raise ValueError(\'Trial calibration mismatch\')\n        y=frames[\'test\'].label.to_numpy()\n        check_metrics(y,p,record[\'raw_threshold\'],record[\'raw_test\'])\n        check_metrics(y,q,record[\'threshold\'],record[\'calibrated_test\'])\n        check_metrics(y,p,.5,record[\'raw_test_at_0_5\'])\n        raw_test[trial]=p\n    means={a:float(np.mean(values)) for a,values in selection_by_arch.items()}\n    chosen=sorted(means,key=lambda a:(-means[a],a))[0]\n    if config[\'architecture\']!=chosen:\n        raise ValueError(\'Architecture selection mismatch\')\n    trials=[f\'{chosen}_seed_{s}\' for s in [42,123,2026]]\n    if set(config[\'selected_checkpoints\'])!={t+\'/best_weights.pt\' for t in trials}:\n        raise ValueError(\'Selected ensemble membership mismatch\')\n    for architecture,mean in means.items():\n        close(config[\'selection_architecture_mean_auroc\'][architecture],mean,\'architecture mean\')\n    for split,source in [(\'selection\',raw_selection),(\'test\',raw_test)]:\n        expected_p=temperature_scale(np.mean([source[t] for t in trials],axis=0),config[\'temperature\'])\n        actual_p=scores(root/f\'{split}_predictions.csv\',frames[split],config[\'threshold\'])\n        if not np.allclose(actual_p,expected_p,atol=1e-10,rtol=1e-8):\n            raise ValueError(\'Ensemble reconstruction mismatch: \'+split)\n    metrics=json.loads((root/\'metrics.json\').read_text())\n    check_metrics(frames[\'test\'].label.to_numpy(),actual_p,config[\'threshold\'],metrics[\'test\'])\n    return dict(basic, trials_verified=9, selected_architecture=chosen,\n                ensemble_reconstructed=True, all_reported_test_metrics_verified=True,\n                limitation=\'Verifies internal artifact consistency, not independent patient identity or clinical validity\')\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'directory\',type=Path)\n    args=parser.parse_args()\n    print(json.dumps(validate_benchmark(args.directory),indent=2))\n', 'verify_exported_model.py': '"""Independently reload exported checkpoints and reproduce the entire test cohort on CPU."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageOps\nimport torch\nfrom torchvision import models, transforms\n\nfrom predict import load_config\nfrom validate_benchmark import validate_benchmark, temperature_scale\n\n\ndef model_for(architecture):\n    if architecture==\'efficientnet_b0\':\n        model=models.efficientnet_b0(weights=None)\n        model.classifier[1]=torch.nn.Linear(model.classifier[1].in_features,1)\n    elif architecture==\'densenet121\':\n        model=models.densenet121(weights=None)\n        model.classifier=torch.nn.Linear(model.classifier.in_features,1)\n    elif architecture==\'resnet18\':\n        model=models.resnet18(weights=None)\n        model.fc=torch.nn.Linear(model.fc.in_features,1)\n    else: raise ValueError(\'Unsupported architecture\')\n    return model\n\n\ndef verify(model_dir,data_root,output):\n    model_dir,data_root,output=map(Path,[model_dir,data_root,output])\n    output.mkdir(parents=True,exist_ok=True)\n    consistency=validate_benchmark(model_dir)\n    print(\'Saved artifact consistency passed\',flush=True)\n    config,paths=load_config(model_dir); size=config.get(\'image_size\',224)\n    frame=pd.read_csv(model_dir/\'test_predictions.csv\').sort_values(\'sha256\').reset_index(drop=True)\n    transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])\n    # Cache only the 618 test tensors, keeping source identity checks independent of prediction values.\n    images=[]\n    for row in frame.itertuples():\n        path=data_root/row.original_split/(\'NORMAL\' if row.y_true==0 else \'PNEUMONIA\')/row.filename\n        with Image.open(path) as image:\n            gray=image.convert(\'L\')\n            digest=hashlib.sha256(str(gray.size).encode()+gray.tobytes()).hexdigest()\n            if digest!=row.sha256: raise ValueError(\'Source image hash mismatch: \'+row.filename)\n            resized=ImageOps.pad(image.convert(\'RGB\'),(size,size),method=Image.Resampling.BICUBIC,color=0)\n            images.append(transform(resized))\n    print(\'Verified decoded pixel identity for\',len(images),\'test images\',flush=True)\n    torch.set_num_threads(2)\n    checkpoint_records=[]\n    results=json.loads((model_dir/\'all_trial_results.json\').read_text())\n    for record in results:\n        path=model_dir/record[\'trial\']/\'best_weights.pt\'\n        model=model_for(record[\'architecture\'])\n        model.load_state_dict(torch.load(path,map_location=\'cpu\',weights_only=True),strict=True)\n        model.eval()\n        with torch.inference_mode(): prediction=model(torch.stack(images[:2]))\n        if prediction.shape!=(2,1) or not torch.isfinite(prediction).all():\n            raise ValueError(\'Checkpoint forward failure: \'+record[\'trial\'])\n        checkpoint_records.append(dict(trial=record[\'trial\'],sha256=hashlib.sha256(path.read_bytes()).hexdigest(),\n                                       strict_load_and_forward=True))\n        print(\'Checkpoint passed:\',record[\'trial\'],flush=True)\n        del model\n    member_scores=[]\n    for path in paths:\n        model=model_for(config[\'architecture\'])\n        model.load_state_dict(torch.load(path,map_location=\'cpu\',weights_only=True),strict=True)\n        model.eval(); predictions=[]\n        with torch.inference_mode():\n            for start in range(0,len(images),16):\n                batch=torch.stack(images[start:start+16])\n                predictions.extend(torch.sigmoid(model(batch).flatten()).numpy())\n        member_scores.append(np.asarray(predictions,dtype=float)); del model\n        print(\'Completed CPU test inference:\',path.parent.name,flush=True)\n    raw=np.mean(member_scores,axis=0); calibrated=temperature_scale(raw,config.get(\'temperature\',1.))\n    difference=np.abs(calibrated-frame.probability.to_numpy())\n    predictions=(calibrated>=config[\'threshold\']).astype(int)\n    changed=int(np.sum(predictions!=frame.prediction.to_numpy()))\n    passed=bool(np.isfinite(calibrated).all() and difference.max()<=1e-4 and changed==0)\n    frame.assign(cpu_probability=calibrated,cpu_prediction=predictions,absolute_score_difference=difference).to_csv(output/\'cpu_reproduction.csv\',index=False)\n    report=dict(status=\'passed\' if passed else \'numerical discrepancy requires review\',\n        images=len(frame),image_size=size,architecture=config[\'architecture\'],\n        max_absolute_score_difference=float(difference.max()),mean_absolute_score_difference=float(difference.mean()),\n        changed_predictions=changed,score_tolerance=1e-4,artifact_consistency=consistency,\n        checkpoints=checkpoint_records,torch_version=torch.__version__,\n        interpretation=\'Engineering reproduction on the same test cohort, not independent external or clinical validation\')\n    (output/\'summary.json\').write_text(json.dumps(report,indent=2))\n    print(\'DONE\',json.dumps(report),flush=True)\n    if not passed: raise RuntimeError(\'CPU numerical reproduction requires review; see saved report\')\n    return report\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--model-dir\',type=Path,required=True)\n    parser.add_argument(\'--data-root\',type=Path,required=True)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    args=parser.parse_args(); verify(args.model_dir,args.data_root,args.output)\n', 'verify_calibration_fit.py': '"""Recompute calibration-only scores on CPU; never replace the frozen model config."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image, ImageOps\nfrom scipy.optimize import minimize_scalar\nfrom scipy.special import expit, logit\nfrom sklearn.metrics import roc_curve\nfrom torchvision import transforms\n\nfrom predict import load_config\nfrom verify_exported_model import model_for\n\n\ndef fit_calibration(y, raw):\n    z=logit(np.clip(np.asarray(raw,dtype=float),1e-7,1-1e-7))\n    y=np.asarray(y,dtype=int)\n    def objective(log_temperature):\n        scaled=z/np.exp(log_temperature)\n        return float(np.mean(np.logaddexp(0,scaled)-y*scaled))\n    fit=minimize_scalar(objective,bounds=(-3,3),method=\'bounded\')\n    if not fit.success: raise ValueError(\'Temperature optimization failed\')\n    temperature=float(np.exp(fit.x)); scores=expit(z/temperature)\n    fpr,tpr,thresholds=roc_curve(y,scores)\n    finite=np.isfinite(thresholds)\n    threshold=float(thresholds[finite][np.argmax((tpr-fpr)[finite])])\n    return temperature,threshold,scores\n\n\ndef calibration_frame(model_dir):\n    model_dir=Path(model_dir)\n    frame=pd.read_csv(model_dir/\'calibration_predictions.csv\').sort_values(\'sha256\').reset_index(drop=True)\n    manifest=pd.read_csv(model_dir/\'split_manifest.csv\')\n    expected=manifest.loc[manifest[\'split\']==\'calibration\']\n    expected_ids=set(expected[[\'path\',\'label\',\'patient\',\'sha256\']].itertuples(index=False,name=None))\n    observed_ids=set(frame[[\'path\',\'y_true\',\'patient\',\'sha256\']].itertuples(index=False,name=None))\n    if len(frame)!=482 or len(expected)!=482 or frame.sha256.duplicated().any() or expected_ids!=observed_ids:\n        raise ValueError(\'Expected the frozen 482-image calibration partition\')\n    return frame\n\n\ndef verify(model_dir,data_root,output):\n    model_dir,data_root,output=map(Path,(model_dir,data_root,output))\n    output.mkdir(parents=True,exist_ok=True)\n    config,checkpoints=load_config(model_dir)\n    size=config.get(\'image_size\',224)\n    frame=calibration_frame(model_dir)\n    torch.set_num_threads(2)\n    transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])\n    images=[]\n    for row in frame.itertuples():\n        path=data_root/row.original_split/(\'NORMAL\' if row.y_true==0 else \'PNEUMONIA\')/row.filename\n        with Image.open(path) as image:\n            gray=image.convert(\'L\')\n            if hashlib.sha256(str(gray.size).encode()+gray.tobytes()).hexdigest()!=row.sha256:\n                raise ValueError(\'Calibration pixel identity mismatch\')\n            images.append(transform(ImageOps.pad(image.convert(\'RGB\'),(size,size),method=Image.Resampling.BICUBIC,color=0)))\n    members=[];hashes=[]\n    for checkpoint in checkpoints:\n        model=model_for(config[\'architecture\'])\n        model.load_state_dict(torch.load(checkpoint,map_location=\'cpu\',weights_only=True),strict=True)\n        model.eval();values=[]\n        with torch.inference_mode():\n            for start in range(0,len(images),16):\n                values.extend(torch.sigmoid(model(torch.stack(images[start:start+16])).flatten()).numpy())\n        members.append(np.asarray(values,dtype=float))\n        hashes.append(dict(checkpoint=checkpoint.relative_to(model_dir).as_posix(),sha256=hashlib.sha256(checkpoint.read_bytes()).hexdigest()))\n        print(\'CALIBRATION_MEMBER_DONE\',size,checkpoint.parent.name,flush=True)\n        del model\n    raw=np.mean(members,axis=0)\n    frozen_scores=expit(logit(np.clip(raw,1e-7,1-1e-7))/config[\'temperature\'])\n    temperature,threshold,refit_scores=fit_calibration(frame.y_true,raw)\n    error=float(np.max(np.abs(frozen_scores-frame.probability.to_numpy())))\n    changed=int(np.sum((frozen_scores>=config[\'threshold\']).astype(int)!=frame.prediction.to_numpy()))\n    relative_temperature_error=abs(temperature/config[\'temperature\']-1)\n    threshold_error=abs(threshold-config[\'threshold\'])\n    # Set before execution; numerical agreement, not a new model-selection criterion.\n    passed=error<=1e-4 and changed==0 and relative_temperature_error<=1e-3 and threshold_error<=1e-4\n    export=frame.copy()\n    for checkpoint,values in zip(checkpoints,members): export[\'cpu_raw_\'+checkpoint.parent.name]=values\n    export[\'cpu_raw_mean\']=raw;export[\'cpu_frozen_temperature_score\']=frozen_scores\n    export[\'cpu_refitted_temperature_score\']=refit_scores\n    export.to_csv(output/\'calibration_cpu_scores.csv\',index=False)\n    report=dict(status=\'passed\' if passed else \'numerical discrepancy requires review\',image_size=size,\n        architecture=config[\'architecture\'],images=len(frame),checkpoints=hashes,\n        saved_temperature=config[\'temperature\'],cpu_refitted_temperature=temperature,\n        relative_temperature_error=relative_temperature_error,saved_threshold=config[\'threshold\'],\n        cpu_refitted_threshold=threshold,absolute_threshold_error=threshold_error,\n        max_frozen_score_difference=error,changed_frozen_calibration_predictions=changed,\n        tolerances=dict(score_absolute=1e-4,temperature_relative=1e-3,threshold_absolute=1e-4),\n        test_predictions_accessed=False,frozen_config_changed=False,torch_version=torch.__version__,\n        interpretation=\'Calibration-only numerical reproduction from original pixels and saved checkpoints; no clinical or external validation. Fresh CPU scores are additional evidence, not original GPU calibration scores.\')\n    (output/\'summary.json\').write_text(json.dumps(report,indent=2))\n    print(\'CALIBRATION_VERIFICATION\',json.dumps(report),flush=True)\n    return report\n'}
for name,source in sources.items(): (dest/name).write_text(source)
sys.path.insert(0,str(dest))
from verify_calibration_fit import verify
root=Path('/kaggle/input')
roots=sorted({p.parent for p in root.rglob('train') if (p/'NORMAL').is_dir() and (p.parent/'test'/'PNEUMONIA').is_dir()},key=lambda p:(len(p.parts),str(p)))
if not roots: raise ValueError('Attach original chest X-ray data')
reports=[]
for size,name in [(224,'pneumonia_benchmark_outputs'),(320,'pneumonia_resolution320_outputs')]:
    matches=[p.parent for p in root.rglob('inference_config.json') if p.parent.name==name]
    if len(matches)!=1: raise ValueError('Attach exactly one completed output: '+name)
    reports.append(verify(matches[0],roots[0],Path('/kaggle/working/calibration_reproduction_outputs')/str(size)))
print('CALIBRATION_ALL_DONE',json.dumps(reports),flush=True)
if any(r['status']!='passed' for r in reports): raise RuntimeError('Review numerical discrepancies; frozen configs unchanged')
